In [11]:
# data_utils.py
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    """
    自定义数据集类
    """
    def __init__(self, features, labels):
        super().__init__()
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature_vector = self.features[idx]
        label = self.labels[idx] # 现在 label 已经是整数类型
        # 可以选择在创建张量时再次确认，但通常在 __init__ 中处理更好
        return {'data': torch.FloatTensor(feature_vector), 'target': torch.tensor(label, dtype=torch.long)}

def get_fixed_split_dataloaders(data_path: str,
                                batch_size: int,
                                logs_dir: str, # 用于保存分割信息的日志目录
                                label_column: str = "dead",
                                train_ratio: float = 0.7,
                                random_seed: int = 256,
                                save_split_info: bool = True):
    """
    加载数据，进行固定的训练/测试集分割，并返回DataLoaders。
    (函数内容不变)
    """
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"数据文件未找到: {data_path}")

    # --- 修改此行 ---
    train_percentage = train_ratio * 100
    test_percentage = (1 - train_ratio) * 100
    print(f"从 {data_path} 加载数据并进行 {train_percentage:.0f}%/{test_percentage:.0f}% 分割...")
    # --- 修改结束 ---

    original_df = pd.read_csv(data_path)
    # 1. 读取原始数据
    original_df = pd.read_csv(data_path)

    if label_column not in original_df.columns:
        raise ValueError(f"标签列 '{label_column}' 不在数据文件中。可用列: {original_df.columns.tolist()}")

    # 2. 分割特征 (X) 和标签 (y)
    y_all_np = original_df[label_column].to_numpy()
    X_all_df = original_df.drop(label_column, axis=1)
    feature_names = X_all_df.columns.tolist()
    num_features = len(feature_names)
    X_all_np = X_all_df.to_numpy()

    # 3. 基于固定的随机种子打乱索引，然后分割索引
    np.random.seed(random_seed)
    num_samples = len(original_df)
    indices = np.arange(num_samples)
    np.random.shuffle(indices) # 这个打乱是固定的

    split_point = int(num_samples * train_ratio)
    train_indices = indices[:split_point]
    test_indices = indices[split_point:]

    # 4. 使用索引获取训练集和测试集
    X_train_np = X_all_np[train_indices]
    y_train_np = y_all_np[train_indices]
    X_test_np = X_all_np[test_indices]
    y_test_np = y_all_np[test_indices]

    num_train_samples = len(y_train_np)
    num_test_samples = len(y_test_np)

    print(f"  - 特征数量: {num_features}")
    print(f"  - 总样本数: {num_samples}")
    print(f"  - 训练样本数: {num_train_samples}")
    print(f"  - 测试样本数: {num_test_samples}")

    # 5. (可选) 保存分割后的数据信息到日志目录
    if save_split_info:
        os.makedirs(logs_dir, exist_ok=True) # 确保目录存在
        train_df_to_save = pd.DataFrame(X_train_np, columns=feature_names)
        train_df_to_save[label_column] = y_train_np
        train_df_to_save.to_csv(os.path.join(logs_dir, f"split_train_data_seed{random_seed}.csv"), index=False)

        test_df_to_save = pd.DataFrame(X_test_np, columns=feature_names)
        test_df_to_save[label_column] = y_test_np
        test_df_to_save.to_csv(os.path.join(logs_dir, f"split_test_data_seed{random_seed}.csv"), index=False)
        print(f"  - 分割后的训练集和测试集已保存至: {logs_dir}")

    # 6. 创建 PyTorch Dataset 和 DataLoader
    train_dataset = CustomDataset(features=X_train_np, labels=y_train_np)
    test_dataset = CustomDataset(features=X_test_np, labels=y_test_np)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # 训练时打乱
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) # 测试时不打乱

    return train_loader, test_loader, num_train_samples, num_test_samples, num_features


if __name__ == '__main__':
    # 测试 data_utils.py
    base_data_dir = "/mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/data"

    #data_path_36_factors_mimic3 = os.path.join(base_data_dir, "36/mimic4_smote_36.csv")
    data_path_8_factors_local = os.path.join(base_data_dir, "8/mimic4_smote_8.csv")

    # --- 修改开始 ---
    # 如果在Jupyter Notebook中，且Notebook与最终的.py文件在同一目录
    try:
        current_script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError: # __file__ is not defined, so we are likely in an interactive environment
        print("Warning: __file__ is not defined. Assuming interactive environment (e.g., Jupyter Notebook).")
        print(f"Using current working directory: {os.getcwd()} as script directory.")
        current_script_dir = os.getcwd() # 获取当前Notebook的工作目录
    # --- 修改结束 ---

    test_logs_dir = os.path.join(current_script_dir, "temp_data_utils_logs")
    os.makedirs(test_logs_dir, exist_ok=True)
    test_logs_dir = os.path.join(current_script_dir, "temp_data_utils_logs")
    os.makedirs(test_logs_dir, exist_ok=True) # 确保临时日志主目录存在

    print("\n--- 测试MIMIC-III 36因子数据 ---")
    if os.path.exists(data_path_36_factors_mimic3):
        print(f"找到数据: {data_path_36_factors_mimic3}")
        train_dl_36, test_dl_36, n_train_36, n_test_36, n_feat_36 = get_fixed_split_dataloaders(
            data_path=data_path_36_factors_mimic3,
            batch_size=32,
            logs_dir=os.path.join(test_logs_dir, "mimic4_36_factors_split"), # 为每个测试创建一个子目录
            label_column="dead",
            save_split_info=True # 确保在测试时保存分割信息
        )
        print(f"MIMIC-III 36因子 - 特征数: {n_feat_36}, 训练样本: {n_train_36}, 测试样本: {n_test_36}")
        print(f"  训练批次数: {len(train_dl_36)}, 测试批次数: {len(test_dl_36)}")
        # 检查一个批次
        try:
            batch_data_train = next(iter(train_dl_36))
            print(f"  一个训练批次数据形状: {batch_data_train['data'].shape}, 标签形状: {batch_data_train['target'].shape}")
            batch_data_test = next(iter(test_dl_36))
            print(f"  一个测试批次数据形状: {batch_data_test['data'].shape}, 标签形状: {batch_data_test['target'].shape}")
        except StopIteration:
            print("  数据加载器为空，无法获取批次。")
    else:
        print(f"警告: MIMIC-III 36因子数据文件未找到于 {data_path_36_factors_mimic3}")

    print("\n--- 测试Local 8因子数据 ---")
    if os.path.exists(data_path_8_factors_local):
        print(f"找到数据: {data_path_8_factors_local}")
        train_dl_8, test_dl_8, n_train_8, n_test_8, n_feat_8 = get_fixed_split_dataloaders(
            data_path=data_path_8_factors_local,
            batch_size=32,
            logs_dir=os.path.join(test_logs_dir, "mimic4_8_factors_split"), # 为每个测试创建一个子目录
            label_column="dead", # 假设这个8因子数据也有名为 "dead" 的标签列
            save_split_info=True
        )
        print(f"Local 8因子 - 特征数: {n_feat_8}, 训练样本: {n_train_8}, 测试样本: {n_test_8}")
        print(f"  训练批次数: {len(train_dl_8)}, 测试批次数: {len(test_dl_8)}")
        try:
            batch_data_train = next(iter(train_dl_8))
            print(f"  一个训练批次数据形状: {batch_data_train['data'].shape}, 标签形状: {batch_data_train['target'].shape}")
        except StopIteration:
            print("  数据加载器为空，无法获取批次。")
    else:
        print(f"警告: Local 8因子数据文件未找到于 {data_path_8_factors_local}")

    # 你可以取消注释并添加其他数据文件的测试：
    # print("\n--- 测试MIMIC-IV 36因子数据 ---")
    # if os.path.exists(data_path_36_factors_mimic4):
    #     # ... 类似上面的测试代码 ...
    # else:
    #     print(f"警告: MIMIC-IV 36因子数据文件未找到于 {data_path_36_factors_mimic4}")

Using current working directory: /mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/model2 as script directory.

--- 测试MIMIC-III 36因子数据 ---
找到数据: /mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/data/36/mimic4_smote_36.csv
从 /mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/data/36/mimic4_smote_36.csv 加载数据并进行 70%/30% 分割...
  - 特征数量: 36
  - 总样本数: 6465
  - 训练样本数: 4525
  - 测试样本数: 1940
  - 分割后的训练集和测试集已保存至: /mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/model2/temp_data_utils_logs/mimic4_36_factors_split
MIMIC-III 36因子 - 特征数: 36, 训练样本: 4525, 测试样本: 1940
  训练批次数: 142, 测试批次数: 61
  一个训练批次数据形状: torch.Size([32, 36]), 标签形状: torch.Size([32])
  一个测试批次数据形状: torch.Size([32, 36]), 标签形状: torch.Size([32])

--- 测试Local 8因子数据 ---
找到数据: /mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/data/8/mimic4_smote_8.csv
从 /mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/data/8/mimic4_smote_8.csv 加载数据并进行 70%/30% 分割...
  - 特征数量: 8
  - 总样本数: 6465
  - 训练样本数: 4525
  - 测试样本数: 1940
  - 分割后

/tmp/ipykernel_366/3074884687.py:24: DeprecationWarning: an integer is required (got type numpy.float64).  Implicit conversion to integers using __int__ is deprecated, and may be removed in a future version of Python.
  return {'data': torch.FloatTensor(feature_vector), 'target': torch.tensor(label, dtype=torch.long)}


In [6]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}")
        model = Lstm( # 严格按照原始 train.py 的实例化方式
            factors=num_features,
            batch_size=args.batch_size, # 原始 Lstm 类接收此参数
            drop_ratio=args.drop_ratio,
            device=args.device # 原始 Lstm 类接收此参数
        )
    # ... (其他模型的初始化逻辑保持不变，确保它们也与原始行为一致) ...
    elif args.model_name == "Transformer":
        model = Transformer(input_dim=num_features, model_dim=args.model_dim, depth=args.depth,
                            num_heads=args.num_heads, drop_ratio=args.drop_ratio)
    elif args.model_name == "GRU":
        model = GRU(factors=num_features, batch_size=args.batch_size, num_layers=args.num_layers,
                    drop_ratio=args.drop_ratio, device=args.device)
    elif args.model_name == "GPT":
        model = GPT(hidden_dim=args.model_dim, num_heads=args.num_heads, # 使用model_dim作为hidden_dim
                    num_layers=args.num_layers, dropout=args.drop_ratio, device=args.device)
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")
    return model.to(args.device)

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time
        print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")

        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")

    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")


if __name__ == '__main__':
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic3_36_factors_split', # 预分割数据目录
        '--model_name', 'Transformer',                                       # 模型名称
        # --- Transformer 特定参数 (基于你的hyperParameters默认值和摘要信息) ---
        '--model_dim', '128',       # Transformer 内部维度
        '--depth', '8',             # Transformer 深度 (block数量)
        '--num_heads', '8',         # Transformer 注意力头数
        '--drop_ratio', '0.1',      # Dropout 比率
        # '--drop_path_ratio', '0.1', # 如果你的Transformer类使用它，取消注释并确保initialize_model处理
        # --- 通用参数 (严格参考你提供的原始摘要和hyperParameters) ---
        '--lr', '0.0005',           # 学习率 (与原始摘要一致)
        '--epochs', '1000',         # Epochs (与原始摘要一致)
        '--batch_size', '5000',     # 批量大小 (与原始摘要一致)
        '--output_base_dir', './dl_model_outputs_replication', # 新的输出基础目录
        '--experiment_tag', 'mimic3-36f-transformer-replication', # 实验标签
        '--seed', '42',             # 随机种子 (与原始摘要的seed一致, 如果不同请修改)
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead',   # 标签列名
        # '--num_layers', '2',      # 对于Transformer，depth参数更相关。如果你的Transformer用这个，取消注释
    ]
    main()


--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 1000
  experiment_tag: mimic3-36f-transformer-replication
  label_column: dead
  lr: 0.0005
  model_dim: 128
  model_name: Transformer
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic3_36_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/Transformer_mimic3_36_factors_mimic3-36f-transformer-replication_trainseed42_20250610-031107
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
  - 特征数量: 36
  - 训练样本数: 2667
  - 测试样本数: 1144
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic3_36_factors_split，特征数: 36

--- 模型结构 ---
Transformer(
  (emmb): Sequential(
    (0): Linear(in_features=36, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_

/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


Epoch [001/1000] | Train Loss: 222.6470 | Test Acc: 0.4921 | Test AUC: 0.6182 | Time: 1.04s
  * New best Test AUC: 0.6182 at Epoch 1. Model and DeLong data updated.
Epoch [002/1000] | Train Loss: 228.5657 | Test Acc: 0.5079 | Test AUC: 0.6374 | Time: 0.14s
  * New best Test AUC: 0.6374 at Epoch 2. Model and DeLong data updated.
Epoch [003/1000] | Train Loss: 358.8972 | Test Acc: 0.5079 | Test AUC: 0.6367 | Time: 0.10s
Epoch [004/1000] | Train Loss: 251.4141 | Test Acc: 0.4991 | Test AUC: 0.6297 | Time: 0.12s
Epoch [005/1000] | Train Loss: 219.6642 | Test Acc: 0.4921 | Test AUC: 0.6287 | Time: 0.13s
Epoch [006/1000] | Train Loss: 223.7497 | Test Acc: 0.4921 | Test AUC: 0.5890 | Time: 0.12s
Epoch [007/1000] | Train Loss: 223.2490 | Test Acc: 0.4921 | Test AUC: 0.5883 | Time: 0.11s
Epoch [008/1000] | Train Loss: 220.4099 | Test Acc: 0.5271 | Test AUC: 0.6160 | Time: 0.10s
Epoch [009/1000] | Train Loss: 219.6541 | Test Acc: 0.5079 | Test AUC: 0.6191 | Time: 0.11s
Epoch [010/1000] | Train L

In [12]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}")
        model = Lstm( # 严格按照原始 train.py 的实例化方式
            factors=num_features,
            batch_size=args.batch_size, # 原始 Lstm 类接收此参数
            drop_ratio=args.drop_ratio,
            device=args.device # 原始 Lstm 类接收此参数
        )
    # ... (其他模型的初始化逻辑保持不变，确保它们也与原始行为一致) ...
    elif args.model_name == "Transformer":
        model = Transformer(input_dim=num_features, model_dim=args.model_dim, depth=args.depth,
                            num_heads=args.num_heads, drop_ratio=args.drop_ratio)
    elif args.model_name == "GRU":
        model = GRU(factors=num_features, batch_size=args.batch_size, num_layers=args.num_layers,
                    drop_ratio=args.drop_ratio, device=args.device)
    elif args.model_name == "GPT":
        model = GPT(hidden_dim=args.model_dim, num_heads=args.num_heads, # 使用model_dim作为hidden_dim
                    num_layers=args.num_layers, dropout=args.drop_ratio, device=args.device)
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")
    return model.to(args.device)

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
            f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
            f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")

    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")


if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 Lstm 在 MIMIC-III 36因子数据上 ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic3_36_factors_split', # 预分割数据目录
        '--model_name', 'Lstm',                                              # 模型名称
        # --- Lstm 相关参数 (基于你的hyperParameters和model_LSTM.py分析) ---
        '--drop_ratio', '0.1',      # Dropout 比率 (Lstm类使用)
        '--num_layers', '2',        # 命令行参数，但你的Lstm实现可能不使用它来改变层结构
                                    # 如果Lstm类不接受num_layers, initialize_model会忽略它
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.0005',           # 学习率
        '--epochs', '800',          # Epochs
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 新的输出基础目录
        '--experiment_tag', 'mimic3-36f-lstm-replication',   # 实验标签
        '--seed', '42',             # 随机种子 (假设与原始摘要的seed一致)
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead'    # 标签列名
        # --- Lstm通常不直接使用的参数 (可以保留，argparse会处理，initialize_model会忽略) ---
        # '--model_dim', '128',
        # '--depth', '8',
        # '--num_heads', '8',
    ]
    # --------------------------------------------------------------------

    # 调用你的主函数
    main()


--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 800
  experiment_tag: mimic3-36f-lstm-replication
  label_column: dead
  lr: 0.0005
  model_dim: 128
  model_name: Lstm
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic3_36_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/Lstm_mimic3_36_factors_mimic3-36f-lstm-replication_trainseed42_20250610-032242
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
  - 特征数量: 36
  - 训练样本数: 2667
  - 测试样本数: 1144
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic3_36_factors_split，特征数: 36
初始化 Lstm模型: factors=36, batch_size=5000, drop_ratio=0.1

--- 模型结构 ---
Lstm(
  (backbone1): LSTM(36, 864)
  (backbone2): LSTM(864, 864)
  (backbone3): LSTM(864, 144)
  (head): Sequential(
  

/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.5937 at Epoch 1. Model and DeLong data updated.
  * New best Test AUC: 0.6163 at Epoch 2. Model and DeLong data updated.
  * New best Test AUC: 0.6376 at Epoch 3. Model and DeLong data updated.
  * New best Test AUC: 0.6414 at Epoch 5. Model and DeLong data updated.
  * New best Test AUC: 0.6450 at Epoch 6. Model and DeLong data updated.
  * New best Test AUC: 0.6518 at Epoch 7. Model and DeLong data updated.
  * New best Test AUC: 0.6591 at Epoch 9. Model and DeLong data updated.
  * New best Test AUC: 0.6595 at Epoch 10. Model and DeLong data updated.
  * New best Test AUC: 0.6660 at Epoch 12. Model and DeLong data updated.
  * New best Test AUC: 0.6699 at Epoch 13. Model and DeLong data updated.
  * New best Test AUC: 0.6701 at Epoch 14. Model and DeLong data updated.
  * New best Test AUC: 0.6747 at Epoch 16. Model and DeLong data updated.
  * New best Test AUC: 0.6860 at Epoch 17. Model and DeLong data updated.
  * New best Test AUC: 0.6903 at Epoch 20. Mo

In [ ]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}")
        model = Lstm( # 严格按照原始 train.py 的实例化方式
            factors=num_features,
            batch_size=args.batch_size, # 原始 Lstm 类接收此参数
            drop_ratio=args.drop_ratio,
            device=args.device # 原始 Lstm 类接收此参数
        )
    # ... (其他模型的初始化逻辑保持不变，确保它们也与原始行为一致) ...
    elif args.model_name == "Transformer":
        model = Transformer(input_dim=num_features, model_dim=args.model_dim, depth=args.depth,
                            num_heads=args.num_heads, drop_ratio=args.drop_ratio)
    elif args.model_name == "GRU":
        model = GRU(factors=num_features, batch_size=args.batch_size, num_layers=args.num_layers,
                    drop_ratio=args.drop_ratio, device=args.device)
    elif args.model_name == "GPT":
        model = GPT(hidden_dim=args.model_dim, num_heads=args.num_heads, # 使用model_dim作为hidden_dim
                    num_layers=args.num_layers, dropout=args.drop_ratio, device=args.device)
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")
    return model.to(args.device)

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time

        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")

    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")


if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 GRU 在 MIMIC-III 36因子数据上 ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic3_36_factors_split', # 预分割数据目录
        '--model_name', 'GRU',                                               # 模型名称
        # --- GRU 特定参数 (基于你的hyperParameters和摘要信息) ---
        '--num_layers', '2',        # GRU 的层数 (与原始摘要一致)
        '--drop_ratio', '0.1',      # Dropout 比率 (与原始摘要一致)
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.0005',           # 学习率
        '--epochs', '1200',         # Epochs (与原始摘要一致，注意与Lstm/Transformer不同)
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 新的输出基础目录
        '--experiment_tag', 'mimic3-36f-gru-replication',    # 实验标签
        '--seed', '42',             # 随机种子 (假设与原始摘要的seed一致)
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead'    # 标签列名
        # --- GRU通常不直接使用的参数 (可以保留，argparse会处理，initialize_model会忽略) ---
        # '--model_dim', '128',
        # '--depth', '8',
        # '--num_heads', '8',
    ]
    # --------------------------------------------------------------------

    # 调用你的主函数
    main()


--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 1200
  experiment_tag: mimic3-36f-gru-replication
  label_column: dead
  lr: 0.0005
  model_dim: 128
  model_name: GRU
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic3_36_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/GRU_mimic3_36_factors_mimic3-36f-gru-replication_trainseed42_20250610-032613
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
  - 特征数量: 36
  - 训练样本数: 2667
  - 测试样本数: 1144
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic3_36_factors_split，特征数: 36

--- 模型结构 ---
GRU(
  (gru): GRU(36, 576, num_layers=2)
  (head1): Sequential(
    (0): Linear(in_features=36, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False

/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.6292 at Epoch 2. Model and DeLong data updated.
Epoch [002/1200] | Train Loss: 269.5751 | Test Acc: 0.4921 | Test AUC: 0.6292 | Time: 0.08s
  * New best Test AUC: 0.6328 at Epoch 6. Model and DeLong data updated.
Epoch [006/1200] | Train Loss: 217.0624 | Test Acc: 0.4921 | Test AUC: 0.6328 | Time: 0.07s
  * New best Test AUC: 0.6384 at Epoch 7. Model and DeLong data updated.
Epoch [007/1200] | Train Loss: 214.9508 | Test Acc: 0.4921 | Test AUC: 0.6384 | Time: 0.07s
  * New best Test AUC: 0.6410 at Epoch 8. Model and DeLong data updated.
Epoch [008/1200] | Train Loss: 214.2660 | Test Acc: 0.4921 | Test AUC: 0.6410 | Time: 0.07s
  * New best Test AUC: 0.6430 at Epoch 9. Model and DeLong data updated.
Epoch [009/1200] | Train Loss: 212.4284 | Test Acc: 0.4921 | Test AUC: 0.6430 | Time: 0.20s
  * New best Test AUC: 0.6458 at Epoch 10. Model and DeLong data updated.
Epoch [010/1200] | Train Loss: 210.1877 | Test Acc: 0.4921 | Test AUC: 0.6458 | Time: 0.07s
  * New b

In [3]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}")
        model = Lstm( # 严格按照原始 train.py 的实例化方式
            factors=num_features,
            batch_size=args.batch_size, # 原始 Lstm 类接收此参数
            drop_ratio=args.drop_ratio,
            device=args.device # 原始 Lstm 类接收此参数
        )
    # ... (其他模型的初始化逻辑保持不变，确保它们也与原始行为一致) ...
    elif args.model_name == "Transformer":
        model = Transformer(input_dim=num_features, model_dim=args.model_dim, depth=args.depth,
                            num_heads=args.num_heads, drop_ratio=args.drop_ratio)
    elif args.model_name == "GRU":
        model = GRU(factors=num_features, batch_size=args.batch_size, num_layers=args.num_layers,
                    drop_ratio=args.drop_ratio, device=args.device)
    elif args.model_name == "GPT":
        gpt_input_dim = num_features
        gpt_hidden_dim_internal = num_features # 为了匹配原始 train.py 的 hidden_dim=factors
        gpt_num_heads_internal = 4
        gpt_ff_expansion_factor_internal = 128 
        model = GPT(
            input_dim=gpt_input_dim,                   # 传递原始输入维度
            hidden_dim=gpt_hidden_dim_internal,        # 模型内部工作维度
            num_heads=gpt_num_heads_internal,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_ff_expansion_factor_internal, # 使用原始值
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 可以使用默认值或从命令行获取
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")
    return model.to(args.device)

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")


if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 GPT 在 MIMIC-III 36因子数据上 ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic3_36_factors_split', # 预分割数据目录
        '--model_name', 'GPT',                                               # 模型名称
        # --- GPT 特定参数 (严格基于原始 train.py 中的硬编码和摘要信息) ---
        # hidden_dim 将从 num_features (即原始的 factors=36) 在 initialize_model 中设置
        # num_heads 将在 initialize_model 中硬编码为 4
        # num_layers 将从命令行参数获取，默认为 2 (与原始硬编码一致)
        '--num_layers', '2',
        '--drop_ratio', '0.1',      # Dropout 比率 (与原始摘要和代码一致)
        # ff_expansion_factor=128 这个参数如果GPT类需要，必须在initialize_model中硬编码或添加命令行参数
        
        # --- 如果想让命令行能控制GPT的model_dim和num_heads，而不是用原始的硬编码/factors值 ---
        # '--model_dim', '36',      # 明确设置 model_dim (会被用作 hidden_dim，与原始 factors 一致)
        # '--num_heads', '4',       # 明确设置 num_heads (与原始硬编码一致)
        # --- 我们将在 initialize_model 中处理这些特殊情况，使其行为与原始 train.py 一致 ---

        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.0005',           # 学习率
        '--epochs', '2000',         # Epochs (与原始摘要一致)
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 新的输出基础目录
        '--experiment_tag', 'mimic3-36f-gpt-replication',    # 实验标签
        '--seed', '42',             # 随机种子 (假设与原始摘要的seed一致)
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead'    # 标签列名

        # --- 以下参数如果命令行中提供了，initialize_model for GPT 可能会覆盖或有特殊处理 ---
        # '--model_dim', '128', # 原始 train.py GPT 初始化时 hidden_dim 用了 factors (36)
        # '--num_heads', '8',   # 原始 train.py GPT 初始化时 num_heads 硬编码为 4
    ]

    main()


--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 2000
  experiment_tag: mimic3-36f-gpt-replication
  label_column: dead
  lr: 0.0005
  model_dim: 128
  model_name: GPT
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic3_36_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/GPT_mimic3_36_factors_mimic3-36f-gpt-replication_trainseed42_20250610-034126
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic3_36_factors_split/split_test_data_seed256.csv
  - 特征数量: 36
  - 训练样本数: 2667
  - 测试样本数: 1144
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic3_36_factors_split，特征数: 36

--- 模型结构 ---
GPT(
  (input_projection): Linear(in_features=36, out_features=36, bias=True)
  (pos_embedding): Embedding(512, 36)
  (dropout): Dropout(p=0.1, inplace=False)
  (decoders): Modul

/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.4953 at Epoch 3. Model and DeLong data updated.
Epoch [003/2000] | Train Loss: 1081.7866 | Test Acc: 0.4921 | Test AUC: 0.4953 | Time: 0.07s
  * New best Test AUC: 0.4988 at Epoch 4. Model and DeLong data updated.
Epoch [004/2000] | Train Loss: 314.7752 | Test Acc: 0.4930 | Test AUC: 0.4988 | Time: 0.07s
  * New best Test AUC: 0.5012 at Epoch 5. Model and DeLong data updated.
Epoch [005/2000] | Train Loss: 252.1633 | Test Acc: 0.4939 | Test AUC: 0.5012 | Time: 0.07s
  * New best Test AUC: 0.5038 at Epoch 6. Model and DeLong data updated.
Epoch [006/2000] | Train Loss: 380.8038 | Test Acc: 0.5079 | Test AUC: 0.5038 | Time: 0.07s
  * New best Test AUC: 0.5057 at Epoch 7. Model and DeLong data updated.
Epoch [007/2000] | Train Loss: 426.5196 | Test Acc: 0.4930 | Test AUC: 0.5057 | Time: 0.07s
  * New best Test AUC: 0.5064 at Epoch 8. Model and DeLong data updated.
Epoch [008/2000] | Train Loss: 392.5404 | Test Acc: 0.4930 | Test AUC: 0.5064 | Time: 0.07s
  * New b